# ⚙️ Process Risk Prediction — Random Forest Regressor

This notebook builds a Random Forest Regression model to predict **defect percentage** from fryer process parameters.

**Pipeline:**
1. Generate synthetic fryer process data
2. Exploratory Data Analysis (EDA)
3. Train a Random Forest Regressor with GridSearchCV
4. Evaluate: R² score, MAE, feature importance
5. Save model to `models/rf_model.pkl`

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Reproducibility
np.random.seed(42)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("Libraries loaded successfully!")

## 2. Generate Synthetic Fryer Process Data

We simulate realistic manufacturing data where:
- **fryer_temperature** is the strongest predictor (higher temp → more defects)
- Other features contribute with varying strength
- Gaussian noise adds realism

The defect formula creates nonlinear relationships so the Random Forest can learn complex patterns.

In [ ]:
n_samples = 1000

# Generate feature values with realistic ranges
fryer_temperature = np.random.uniform(150, 200, n_samples)    # °C — main driver
frying_time = np.random.uniform(2, 6, n_samples)              # minutes
moisture_content = np.random.uniform(1, 8, n_samples)          # percentage
slice_thickness = np.random.uniform(1, 3, n_samples)           # mm
oil_quality_index = np.random.uniform(0.5, 1.0, n_samples)    # 0-1 index

# --- Compute defect_percentage ---
# fryer_temperature is the DOMINANT factor (nonlinear: exponential growth above 175°C)
# This mimics real-world behavior where chips burn rapidly at high temperatures

defect_percentage = (
    # Temperature: strongest predictor with exponential effect
    0.40 * ((fryer_temperature - 150) / 50) ** 2 * 100
    # Frying time: longer frying = more defects
    + 0.15 * ((frying_time - 2) / 4) * 100
    # Moisture: low moisture = brittle chips = more defects
    + 0.10 * ((8 - moisture_content) / 7) * 100
    # Slice thickness: thin chips break more
    + 0.05 * ((3 - slice_thickness) / 2) * 100
    # Oil quality: poor oil = more defects
    + 0.15 * ((1.0 - oil_quality_index) / 0.5) * 100
    # Interaction: high temp + long time is especially bad
    + 0.05 * ((fryer_temperature - 150) / 50) * ((frying_time - 2) / 4) * 100
    # Random noise
    + np.random.normal(0, 3, n_samples)
)

# Clip to [0, 100] range
defect_percentage = np.clip(defect_percentage, 0, 100)

# Create DataFrame
df = pd.DataFrame({
    'fryer_temperature': np.round(fryer_temperature, 1),
    'frying_time': np.round(frying_time, 2),
    'moisture_content': np.round(moisture_content, 2),
    'slice_thickness': np.round(slice_thickness, 2),
    'oil_quality_index': np.round(oil_quality_index, 3),
    'defect_percentage': np.round(defect_percentage, 2)
})

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Descriptive statistics
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation = df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, annot=True, fmt='.3f', cmap='RdYlBu_r',
            mask=mask, center=0, square=True,
            linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with defect_percentage:")
print(correlation['defect_percentage'].sort_values(ascending=False).to_string())

In [ ]:
# Distribution of defect_percentage
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')

for i, col in enumerate(df.columns):
    ax = axes[i // 3][i % 3]
    ax.hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: each feature vs defect_percentage
features = ['fryer_temperature', 'frying_time', 'moisture_content', 
             'slice_thickness', 'oil_quality_index']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Features vs Defect Percentage', fontsize=16, fontweight='bold')

for i, feat in enumerate(features):
    axes[i].scatter(df[feat], df['defect_percentage'], alpha=0.3, s=10, color='steelblue')
    axes[i].set_xlabel(feat, fontsize=9)
    axes[i].set_ylabel('Defect %' if i == 0 else '')
    axes[i].set_title(f'r = {df[feat].corr(df["defect_percentage"]):.3f}', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Train Random Forest Regressor

We use `GridSearchCV` to find the best hyperparameters across:
- `n_estimators`: [100, 200, 300]
- `max_depth`: [5, 10, 15, None]
- `min_samples_split`: [2, 5, 10]

In [ ]:
# Prepare features (X) and target (y)
X = df[features]
y = df['defect_percentage']

# Train/test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

In [ ]:
# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],      # Number of trees
    'max_depth': [5, 10, 15, None],        # Maximum tree depth
    'min_samples_split': [2, 5, 10]        # Minimum samples to split a node
}

# Initialize Random Forest
rf = RandomForestRegressor(random_state=42)

# GridSearchCV with 5-fold cross-validation
# Scoring: R² (coefficient of determination)
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,           # Use all CPU cores
    verbose=1,
    return_train_score=True
)

print("Starting GridSearchCV...")
print(f"Total fits: {len(param_grid['n_estimators']) * len(param_grid['max_depth']) * len(param_grid['min_samples_split']) * 5}")

grid_search.fit(X_train, y_train)

print(f"\n✅ GridSearchCV complete!")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV R² score: {grid_search.best_score_:.4f}")

In [ ]:
# Get the best model
best_rf = grid_search.best_estimator_

# Top 5 parameter combinations
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df.nsmallest(5, 'rank_test_score')[[
    'params', 'mean_test_score', 'std_test_score', 'rank_test_score'
]]
print("Top 5 hyperparameter combinations:")
top5

## 5. Model Evaluation

In [ ]:
# Make predictions on test set
y_pred = best_rf.predict(X_test)

# Calculate metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 50)
print("MODEL EVALUATION METRICS")
print("=" * 50)
print(f"R² Score:                {r2:.4f}")
print(f"Mean Absolute Error:     {mae:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f}")
print("=" * 50)

In [ ]:
# Feature Importance Plot
importances = best_rf.feature_importances_
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(features)))
bars = plt.barh(importance_df['Feature'], importance_df['Importance'],
                color=colors, edgecolor='white', height=0.6)
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Random Forest — Feature Importance', fontsize=14, fontweight='bold')

# Add value labels
for bar, val in zip(bars, importance_df['Importance']):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Feature Importance Rankings:")
for _, row in importance_df.sort_values('Importance', ascending=False).iterrows():
    print(f"  {row['Feature']:25s} → {row['Importance']:.4f}")

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.5, s=20, color='steelblue', edgecolors='white', linewidth=0.5)
plt.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Defect %', fontsize=12)
plt.ylabel('Predicted Defect %', fontsize=12)
plt.title(f'Actual vs Predicted (R² = {r2:.4f})', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residual plot
residuals = y_test - y_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
ax1.scatter(y_pred, residuals, alpha=0.5, s=15, color='steelblue')
ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax1.set_xlabel('Predicted Defect %')
ax1.set_ylabel('Residual')
ax1.set_title('Residuals vs Predicted', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Residual distribution
ax2.hist(residuals, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Residual')
ax2.set_ylabel('Count')
ax2.set_title('Residual Distribution', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Save the Model

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save using joblib (efficient for sklearn models with numpy arrays)
model_path = '../models/rf_model.pkl'
joblib.dump(best_rf, model_path)

print(f"✅ Model saved to: {model_path}")
print(f"   Model size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")

# Save the synthetic dataset for reference
df.to_csv('../models/synthetic_data.csv', index=False)
print(f"   Dataset saved to: ../models/synthetic_data.csv")

In [ ]:
# Quick verification — load and test
loaded_model = joblib.load(model_path)

# Test prediction with sample input
sample_input = pd.DataFrame({
    'fryer_temperature': [180],
    'frying_time': [4.0],
    'moisture_content': [3.5],
    'slice_thickness': [2.0],
    'oil_quality_index': [0.75]
})

pred = loaded_model.predict(sample_input)[0]
print(f"\nSample prediction test:")
print(f"  Input: temp=180°C, time=4.0min, moisture=3.5%, thickness=2.0mm, oil=0.75")
print(f"  Predicted defect: {pred:.2f}%")
print(f"\n🎉 Random Forest model training complete!")